# Semana 01 · AgentOps y LLMOps con MLflow

Práctica guiada para observar trazas y evaluación mínima sin depender de un proveedor LLM. Usaremos un agente determinista para aprender el flujo: entrada → ruta → contexto → respuesta → traza → evaluación.


## 0. Dependencias y configuración

Deja `USE_LLM = False`. El objetivo es instrumentar y evaluar, no llamar a un modelo externo.


In [0]:
%pip install -q --upgrade "mlflow[databricks]"
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
from typing import Any

import mlflow

USE_LLM = False
AGENT_VERSION = "v1-deterministic"
EXPERIMENT_NAME = "/Users/my.garciaazuara@gmail.com/semana01-agentops"
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_tags({"course": "operacion-modelos", "week": "01", "agent.version": AGENT_VERSION})

2026/09/24 08:59:59 INFO mlflow.tracking.fluent: Experiment with name '/Users/my.garciaazuara@gmail.com/semana01-agentops' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog
2026/09/24 08:59:59 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o11.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.

## 1. Trazar un agente sencillo

**TODO 1:** completa las reglas de ruta, recuperación y respuesta. Decide qué palabras activan cada ruta, porque esa decisión define el comportamiento observable. Inspecciona `@mlflow.trace`. Verifica que cada pregunta devuelve texto y que en MLflow aparecen spans anidados.


In [0]:
@mlflow.trace
# Hace que MLflow registre la ejecución como una traza o span
def route_question(question: str) -> str:
    text = question.lower()
    if any(word in text for word in ["token", "secreto", "contraseña", "privacidad"]):
        return "security"
    if any(word in text for word in ["producción", "api", "servicio"]):
        return "production"
    return "mlflow"


@mlflow.trace
def retrieve_course_context(topic: str) -> str:
    contexts = {
        "security": "No registres secretos en parámetros, tags ni artefactos.",
        "production": "En producción importan contratos, versionado y monitorización.",
        "mlflow": "Un experimento agrupa runs con parámetros, métricas y artefactos.",
    }
    return contexts[topic]


@mlflow.trace
def course_agent(question: str) -> str:
    route = route_question(question)
    context = retrieve_course_context(route)
    return f"Ruta: {route}. {context}"

questions = [
    "¿Qué objeto agrupa varios runs?",
    "¿Puedo pegar mi token en un parámetro de MLflow?",
    "¿Qué cambia cuando llevo un modelo a producción?",
]
answers = [{"question": q, "answer": course_agent(q)} for q in questions]
display(answers)
assert all("Ruta:" in item["answer"] for item in answers)

2026/09/24 09:00:43 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o19.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumented

answer,question
"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",¿Qué objeto agrupa varios runs?
"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",¿Puedo pegar mi token en un parámetro de MLflow?
"Ruta: production. En producción importan contratos, versionado y monitorización.",¿Qué cambia cuando llevo un modelo a producción?


[Trace(trace_id=tr-c4bb4c2d2c62632ac5478bc7c0196c5a), Trace(trace_id=tr-cdcb9b4a80f7929a9a5d4be2399f39af), Trace(trace_id=tr-fc32b12d00176503970350eb2c233bc1)]

## 2. Inspeccionar una traza

En **Experiments**, abre una traza de `course_agent`. Anota qué ruta siguió la pregunta, qué contexto recuperó y qué cambiarías si una ruta fuese incorrecta.

Para la traza ¿Qué cambia cuando llevo un modelo a producción?: 
course_agent -> route_question -> retrieve_course_context
Recuperó el contexto: En producción importan contratos, versionado y monitorización.
Si la ruta fuera incorrecta tomaría el camino de mlflow y el contexto sería: Un experimento agrupa runs con parámetros, métricas y artefactos.

## 3. Evaluación mínima y transparente

No usamos un juez LLM. Comprobamos una expectativa simple: que la respuesta contenga una palabra esperada.

**TODO 2:** añade al menos dos casos: uno de privacidad/secretos y otro de producción. Importa porque la evaluación debe cubrir comportamientos relevantes. Inspecciona listas de diccionarios y verifica que `evaluation_df` tiene todos los casos.


In [0]:
EVALUATION_DATA = [
    {"question": "¿Qué objeto agrupa varios runs?", "must_include": "experimento"},
    {"question": "¿Puedo guardar una contraseña como tag?", "must_include": "secretos"},
    {"question": "¿Qué debo cuidar en producción?", "must_include": "contratos"},
    {"question": "¿Cómo debo tratar la privacidad de los datos?", "must_include": "secretos"},
    {"question": "¿Qué debo monitorizar en producción?", "must_include": "monitorización"},
]

evaluation_rows = []
for case in EVALUATION_DATA:
    answer = course_agent(case["question"])
    passed = case["must_include"].lower() in answer.lower()
    evaluation_rows.append({**case, "answer": answer, "passed": passed})

evaluation_df = __import__("pandas").DataFrame(evaluation_rows)
display(evaluation_df)
assert evaluation_df["passed"].all(), "Alguna expectativa mínima no se cumple"

question,must_include,answer,passed
¿Qué objeto agrupa varios runs?,experimento,"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",true
¿Puedo guardar una contraseña como tag?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Qué debo cuidar en producción?,contratos,"Ruta: production. En producción importan contratos, versionado y monitorización.",true
¿Cómo debo tratar la privacidad de los datos?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Qué debo monitorizar en producción?,monitorización,"Ruta: production. En producción importan contratos, versionado y monitorización.",true


[Trace(trace_id=tr-7d7faab6b62d0c1837c7b35b02e36c44), Trace(trace_id=tr-14de7d1b878b60138eeb21449ef58689), Trace(trace_id=tr-c1100b3705bddee0c4b7edb90446d94a), Trace(trace_id=tr-ccde083beb1f9d54ae08564714b3edf8), Trace(trace_id=tr-073b3b4d87f71c452d5f7f7ac73bb7be)]

## 4. Registrar un run de evaluación

**TODO 3:** registra el número de casos y el porcentaje que pasa. Importa porque una evaluación sin run se pierde. Inspecciona `mlflow.log_metric` y `mlflow.log_dict`. Verifica que el run contiene el artefacto `evaluation/results.json`.


In [0]:
with mlflow.start_run(run_name=f"agent-evaluation-{AGENT_VERSION}") as run:
    mlflow.set_tags({"agent.version": AGENT_VERSION, "purpose": "guided-agentops-evaluation"})
    mlflow.log_param("case_count", len(evaluation_df))
    mlflow.log_metric("pass_rate", float(evaluation_df["passed"].mean()))
    mlflow.log_dict(evaluation_rows, "evaluation/results.json")
    EVALUATION_RUN_ID = run.info.run_id

print({"evaluation_run_id": EVALUATION_RUN_ID, "pass_rate": float(evaluation_df["passed"].mean())})

2026/09/24 09:13:50 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o27.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumented

{'evaluation_run_id': '11c8b92ec3f74847b5e73675cc00a161', 'pass_rate': 1.0}


## 5. Buscar trazas o runs por versión

**TODO 4:** filtra por `agent.version`. Importa porque un agente cambia rápido y necesitas comparar versiones. Inspecciona `mlflow.search_runs` y, si tu workspace lo permite, `mlflow.search_traces`. Verifica que sólo recuperas la versión `v1-deterministic`.


In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.agent.version = '{AGENT_VERSION}'",
    order_by=["attributes.start_time DESC"],
)
display(runs[["run_id", "tags.agent.version", "metrics.pass_rate"]])
assert set(runs["tags.agent.version"].dropna()) == {AGENT_VERSION}

run_id,tags.agent.version,metrics.pass_rate
11c8b92ec3f74847b5e73675cc00a161,v1-deterministic,1.0
d7275eae72b2434b9b2719749d15290f,v1-deterministic,null


## Entregable

Incluye en la ficha: una captura de una traza, `EVALUATION_RUN_ID`, el `pass_rate` y una frase sobre qué riesgo cubrirías en la siguiente versión del agente.

![image_1790241501993.png](./image_1790241501993.png "image_1790241501993.png")



```
EVALUATION_RUN_ID: 11c8b92ec3f74847b5e73675cc00a161
pass_rate = 1.0
En la siguiente versión cubriría el riesgo de que una pregunta contenga términos asociados a varias rutas, ya que el sistema actual utiliza reglas basadas únicamente en palabras clave.
```
